# AI Identity Transformation Studio — Phase 4: Generative AI Face Transformation

This Google Colab notebook provides GPU-accelerated text-guided and mask-guided face transformation using **Stable Diffusion Inpainting** and **MediaPipe Face Mesh**.

### Objectives:
1. Extract facial boundaries and generate soft inpainting masks.
2. Transform facial appearance into fictional characters (Cyborg, Alien, Anime, Marble Statue, Steampunk).
3. Preserve composition, clothing, and background with high fidelity.
4. Quantify background preservation and processing latency.

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi

In [ ]:
# Step 2: Install required packages
!pip install -q diffusers transformers accelerate mediapipe opencv-python tabulate

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionInpaintPipeline
import mediapipe as mp

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

# Load Stable Diffusion Inpainting pipeline in FP16
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
pipe = pipe.to(device)
if device == 'cuda':
    pipe.enable_attention_slicing()
print('Inpainting pipeline loaded successfully.')

In [ ]:
# Step 3: Facial Mask Generator using MediaPipe Face Mesh
def extract_face_mask(image_bgr, dilate_px=15, feather_k=17, preserve_eyes=False):
    h, w = image_bgr.shape[:2]
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    mp_mesh = mp.solutions.face_mesh
    with mp_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.3) as mesh:
        results = mesh.process(rgb)
        if not results.multi_face_landmarks:
            raise ValueError('No face detected in image.')
        
        pts = []
        for lm in results.multi_face_landmarks[0].landmark:
            pts.append([int(min(lm.x * w, w - 1)), int(min(lm.y * h, h - 1))])
        pts = np.array(pts, dtype=np.int32)
        
    # Face oval hull
    hull = cv2.convexHull(pts)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)
    
    if dilate_px > 0:
        k = dilate_px * 2 + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        mask = cv2.dilate(mask, kernel)
        
    if feather_k > 1:
        mask = cv2.GaussianBlur(mask, (feather_k, feather_k), 0)
        
    return mask
print('Mask generation function ready.')

In [ ]:
# Step 4: Define Transformation Presets
PRESETS = {
    'Cyborg': {
        'prompt': 'high-tech futuristic cybernetic cyborg android face, glowing blue circuitry under chrome skin, metallic plating, polished carbon fiber contours, robotic features, 8k, cinematic',
        'neg': 'human skin, blurry, cartoon, low quality, deformed',
        'strength': 0.80,
        'guidance': 8.0,
    },
    'Alien': {
        'prompt': 'celestial alien humanoid with iridescent bioluminescent skin, subtle organic head ridges, deep obsidian cosmic eyes, ethereal sci-fi aesthetic, masterpiece',
        'neg': 'monstrous, grotesque, human, lowres, blurry',
        'strength': 0.82,
        'guidance': 8.5,
    },
    'Marble Statue': {
        'prompt': 'classical Greco-Roman marble sculpture portrait, carved white Carrara marble with fine mineral veins, museum lighting, chisel texture, antique masterpiece',
        'neg': 'skin, modern clothes, colorful, plastic, blurry',
        'strength': 0.80,
        'guidance': 7.5,
    },
    'Anime Watercolor': {
        'prompt': 'delicate watercolor and ink illustration of a fantasy character face, soft pastel color wash, clean anime line art, expressive eyes, Studio Ghibli style',
        'neg': 'photo, 3d render, plastic, muddy colors, harsh shadows',
        'strength': 0.75,
        'guidance': 7.5,
    },
}
print('Available presets:', list(PRESETS.keys()))

In [ ]:
# Step 5: Execute Transformation on Sample Image
test_image_url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg'
!curl -s -L -o sample.jpg {test_image_url}

img_bgr = cv2.imread('sample.jpg')
mask = extract_face_mask(img_bgr)

pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
pil_mask = Image.fromarray(mask)

selected_style = 'Cyborg'  # Change to 'Alien', 'Marble Statue', or 'Anime Watercolor'
cfg = PRESETS[selected_style]

result_pil = pipe(
    prompt=cfg['prompt'],
    negative_prompt=cfg['neg'],
    image=pil_img,
    mask_image=pil_mask,
    guidance_scale=cfg['guidance'],
    strength=cfg['strength'],
).images[0]

# Display 3-panel comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes[0].imshow(pil_img)
axes[0].set_title('Original Input')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Facial Inpainting Mask')
axes[1].axis('off')

axes[2].imshow(result_pil)
axes[2].set_title(f'Transformed: {selected_style}')
axes[2].axis('off')

plt.tight_layout()
plt.show()